# device initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a

from time import sleep as delay
import datetime
import csv

dm = keithley_dm6500("USB0::0x05E6::0x6500::04651237::INSTR")
ps811 = rigol_dp811a(resource_name="USB0::0x1AB1::0x0E11::DP8H245000515::INSTR")
ps821 = rigol_dp821a(resource_name="USB0::0x1AB1::0x0E11::DP8E261000023::INSTR")

from sc_approval.ch341 import ch341
i2c = ch341(logging=False)

from phy.relay_16ch import relay_box
relay = relay_box(i2c_h=i2c, i2c_a=0x27, logging=False)
relay.init_channels
relay.logging = False

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from sc_approval.ch341 import ch341
from phy.relay_16ch import relay_box

from time import sleep as delay
import datetime
import csv

from PyQt5.QtWidgets import *
from PyQt5.QtGui import *
from PyQt5 import uic
import sys

form_class = uic.loadUiType("./sc_approval/main.ui")[0]

class WindowClass(QMainWindow, form_class) :

    def __init__(self) :

        super().__init__()
        self.setupUi(self)

        self.btn_init.clicked.connect(self.init_equipment)
        self.btn_run.clicked.connect(self.run_10k_test)

        self.txt_log.setFontPointSize(15)
        self.txt_log.append("initialization")
    

    def init_equipment(self):

        self.dm = keithley_dm6500("USB0::0x05E6::0x6500::04651237::INSTR")
        self.ps811 = rigol_dp811a(resource_name="USB0::0x1AB1::0x0E11::DP8H245000515::INSTR")
        self.ps821 = rigol_dp821a(resource_name="USB0::0x1AB1::0x0E11::DP8E261000023::INSTR")
        self.i2c = ch341(logging=False)
        self.relay = relay_box(i2c_h=self.i2c, i2c_a=0x27, logging=False)
        self.relay.init_channels
        self.relay.logging = False
    

    def time_stamp(self):
    
        now   = datetime.datetime.now()
        print("[" + now.strftime("%Y-%m-%d ") + now.strftime("%H:%M:%S" + "] "), end="")


    def output_csv(self, output, filename=None):
    
        # output should be the list type

        with open(f"./log/{filename}.csv", "a", newline="") as f:
            write_handler = csv.writer(f)
            write_handler.writerow(output)
    
    
    def run_10k_test(self):

        self.ps821.ch1.disable
        self.ps811.disable

        delay(3)

        self.ps811.cfg_all = 5, 1
        self.ps811.enable

        count = 1

        v_vext = 20
        v_vout = 4.5

        self.ps = self.ps821.ch1

        self.dm.current_1
        self.relay.init_channels
        self.ps.disable

        self.relay.ch5.enable

        filename = "test"
        self.output_csv(output=["Count", "IQ_VEXT", "IQ_VOUT"], filename=filename)


        for i in range(count):

            #-----------------------------------------------------
            #  IQ_VEXT
            #-----------------------------------------------------
            self.relay.ch3.enable
            # self.time_stamp()
            self.txt_log.append(f"enable relay channel #2 (VEXT)")
            QApplication.processEvents()

            self.ps.cfg_all = v_vext, 0.02 # vext
            self.ps.enable
            # self.time_stamp()
            self.txt_log.append(f"enable power supply to 20V for measuring IQ_VEXT")
            QApplication.processEvents()

            delay(1)
            ret_00h = self.i2c.i2c_read(110, 0)
            # self.time_stamp()
            self.txt_log.append(f"i2c read 00h = {ret_00h:#04x}")
            QApplication.processEvents()

            iq_vext = round(self.dm.current_10E_3 * 1e+6, 3)
            # self.time_stamp()
            self.txt_log.append(f"IQ_VEXT = {iq_vext}uA")
            QApplication.processEvents()

            self.ps.cfg_all = 0.1, 0.02
            self.ps.disable
            # self.time_stamp()
            self.txt_log.append(f"disable the power supply")
            QApplication.processEvents()

            delay(0.5)
            self.relay.ch3.disable
            # self.time_stamp()
            self.txt_log.append(f"disable relay channel #2")
            QApplication.processEvents()
            #-----------------------------------------------------


            #-----------------------------------------------------
            #  IQ_VOUT
            #-----------------------------------------------------
            self.relay.ch1.enable
            # self.time_stamp()
            self.txt_log.append(f"enable relay channel #1 (VOUT)")
            QApplication.processEvents()

            self.ps.cfg_all = v_vout, 0.02 # vout
            self.ps.enable
            # self.time_stamp()
            self.txt_log.append(f"set power supply 4.5V for measuring IQ_VOUT")
            QApplication.processEvents()

            delay(1)
            ret_00h = self.i2c.i2c_read(110, 0)
            # self.time_stamp()
            self.txt_log.append(f"i2c read 00h = {ret_00h:#04x}")
            QApplication.processEvents()

            iq_vout = round(self.dm.current_10E_3 * 1e+6, 3)
            # self.time_stamp()
            self.txt_log.append(f"IQ_VOUT = {iq_vout}uA")
            QApplication.processEvents()

            self.ps.cfg_all = 0.1, 0.02
            self.ps.disable
            # self.time_stamp()
            self.txt_log.append(f"disable the power supply")
            QApplication.processEvents()

            delay(0.5)
            self.relay.ch1.disable
            # self.time_stamp()
            self.txt_log.append(f"disable relay channel #1")
            QApplication.processEvents()
            #-----------------------------------------------------

            self.output_csv(output=[count, iq_vext, iq_vout], filename=filename)
            # self.time_stamp()
            self.txt_log.append(f"store the data into the file")
            QApplication.processEvents()

            self.relay.ch5.disable


if __name__ == "__main__" :

    app = QApplication(sys.argv) 
    myWindow = WindowClass() 
    myWindow.show()
    app.exec_()

# test code

In [ ]:
def time_stamp():
    
    now   = datetime.datetime.now()
    print("[" + now.strftime("%Y-%m-%d ") + now.strftime("%H:%M:%S" + "] "), end="")

In [ ]:
def output_csv(output, filename=None):
    
    # output should be the list type

    with open(f"./log/{filename}.csv", "a", newline="") as f:
        write_handler = csv.writer(f)
        write_handler.writerow(output)

In [ ]:
ps821.ch1.disable
ps811.disable

delay(3)

ps811.cfg_all = 5, 1
ps811.enable

count = 20

v_vext = 20
v_vout = 4

ps = ps821.ch1

dm.current_1
relay.init_channels
ps.disable

relay.ch5.enable

filename = "test"
output_csv(output=["Count", "IQ_VEXT", "IQ_VOUT"], filename=filename)


for i in range(count):

    #-----------------------------------------------------
    #  IQ_VEXT
    #-----------------------------------------------------
    relay.ch3.enable
    time_stamp()
    print(f"enable relay channel #2 (VEXT)")

    ps.cfg_all = v_vext, 0.02 # vext
    ps.enable
    time_stamp()
    print(f"enable power supply to 20V for measuring IQ_VEXT")

    delay(1)
    ret_00h = i2c.i2c_read(110, 0)
    time_stamp()
    print(f"i2c read 00h = {ret_00h:#04x}")

    iq_vext = round(dm.current_10E_3 * 1e+6, 3)
    time_stamp()
    print(f"IQ_VEXT = {iq_vext}uA")

    ps.cfg_all = 0.1, 0.02
    ps.disable
    time_stamp()
    print(f"disable the power supply")

    delay(0.5)
    relay.ch3.disable
    time_stamp()
    print(f"disable relay channel #2")
    #-----------------------------------------------------


    #-----------------------------------------------------
    #  IQ_VOUT
    #-----------------------------------------------------
    relay.ch1.enable
    time_stamp()
    print(f"enable relay channel #1 (VOUT)")

    ps.cfg_all = 4.5, 0.02 # vout
    ps.enable
    time_stamp()
    print(f"set power supply 4.5V for measuring IQ_VOUT")

    delay(1)
    ret_00h = i2c.i2c_read(110, 0)
    time_stamp()
    print(f"i2c read 00h = {ret_00h:#04x}")

    iq_vout = round(dm.current_10E_3 * 1e+6, 3)
    time_stamp()
    print(f"IQ_VOUT = {iq_vout}uA")

    ps.cfg_all = 0.1, 0.02
    ps.disable
    time_stamp()
    print(f"disable the power supply")

    delay(0.5)
    relay.ch1.disable
    time_stamp()
    print(f"disable relay channel #1")
    #-----------------------------------------------------

    output_csv(output=[count, iq_vext, iq_vout], filename=filename)
    time_stamp()
    print(f"store the data into the file")

    relay.ch5.disable